# 1. Information about the submission

## 1.1 Name and number of the assignment

Hallucination Detection in Tool Calling

## 1.2 Student name

Daniil Koblov, Egor Serov

## 1.3 Codalab user ID / nickname / username

Not submitted to Codalab.

## 1.4 Additional comments

We have added our code on the github. There u can  better see all /notebooks/ with all methods and dataset creating. And /results/ folder with whole jsons. Moreover, we added Docker for better reproducability and launch of our code.

The link is: https://github.com/KoblovDA/HallucinationDetection

# 2. Technical Report

## 2.1 Methodology

The task is to detect hallucinations at the character-span level in dialogues that involve a tool call. The approach has three parts: construct a dataset of three hallucination types from the ToolACE corpus, evaluate two off-the-shelf baselines (LettuceDetect and LookBackLens), and report an LLM-as-judge detector that outperforms both. The methodology choices for each part are described below, including the alternatives that were considered and discarded.

### Dataset construction

#### Source filtering

The base dataset is ToolACE (Liu et al., 2024). From its 11,300 dialogues, only the 1,034 dialogues that contain at least one tool execution followed by a natural-language assistant reply are kept. Each is converted to a triple `(query, tool_output, output)` where `query` is the immediately preceding user turn, `tool_output` is the executed tool's JSON response (always valid JSON in ToolACE, since the dataset is synthetic and rule-checked), and `output` is the assistant's reply. The available-tools list is parsed from the dialogue's system prompt and stored as `tools_available`; this is needed for Type 3 injection and is also fed to the LLM-as-judge detector. A statistical pass confirms that 94% of unique APIs are used exactly once in our subset, so any rule must be domain-agnostic — there is no usable "per-tool" prior.

#### Type 1 (contradiction): rule-based pipeline plus LLM fallback

The annotation must produce exact character spans, so a substring-substitution strategy is used wherever possible. Each candidate value from the tool output JSON is matched against the answer text; if a match is found, the substring is replaced with a contradictory alternative. Five strategies are tried in this priority order:

1. **In-sample pool swap.** For tool outputs that return a list of objects (top-N results, listings, search hits), each field repeats with different values across the list. Replacing a value with another value of the same field from the same tool output creates a contradiction whose substituted token is still a "real" value present in the JSON. This is the dominant strategy because the substituted value is not implausible — the detector cannot succeed by "looking for impossible tokens" and must reason about which entity the value belongs to. RAGTruth calls this category "Subtle Conflict".
2. **Type-aware integer / float swap.** For numeric values, the original substring is located with boundary-matching (the adjacent characters must not be digits or a decimal point, to avoid breaking longer numbers such as `0.002`), and a contrastive value is generated by a 3–5× rescale or a random shift.
3. **Type-aware URL swap.** The path or host of a URL is modified.
4. **Type-aware date swap.** Dates appear in the answer in seven distinct formats: ISO verbatim, `Month D, YYYY`, `Month Dth, YYYY`, `Month D`, `Month YYYY`, ISO prefix, and `Dth Month YYYY`. Each tool output ISO date is searched for in all seven formats; on the first match, a new date is generated by a 7–365 day random shift and re-rendered in the same format before substring substitution. This raises date coverage from ~50% (verbatim only) to ~92%.
5. **Cross-sample pool swap.** For scalar fields in repeating tools (or generic semantic fields like `country`, `sentiment`, `sport`), a global value pool aggregated by field name across the corpus is used. This is the strategy that produces the canonical example from the task brief (`country: US → GB`, `sentiment: positive → negative`).

Boolean substitution is excluded from the rule-based pipeline: 94.8% of boolean tool-output values are paraphrased in the answer (`isValid: true` rendered as `"is valid"`, `success: false` as `"failed"`), so substring substitution is not applicable. For the same reason, a blacklist of "dirty" field names — `name, title, id, description, text, symbol, code, condition, type, status, category` — is excluded from in-sample pool swap, because these mix incompatible domains (the `condition` field aggregates Used-Like-New, Partly Cloudy, Heavy Rain, etc.).

The pipeline covers 751 of 1,034 triples (72.6%) with zero broken spans and zero self-swaps. The remaining 283 triples are handled by a single call to Qwen3-235B-A22B-2507 via OpenRouter, with a prompt that asks the model to (a) pick one fact in the answer that is grounded in the tool output, (b) propose a substring-level swap, (c) match word boundaries and case, and (d) avoid one specific grammar trap (`"has been successfully canceled" → "has been failed to cancel"` is ungrammatical; the model is given counterexamples). The proposed `original_substring` is then verified to appear verbatim in the answer and to differ from the `new_substring`, otherwise the call is discarded. Combined coverage reaches 99.8% (1,032 of 1,034). The training split is then expanded to 4,057 samples by drawing up to five distinct injections per source triple, round-robin across strategies for diversity.

#### Type 2 (overgeneration): LLM-only

A rule-based approach is not appropriate here because overgeneration is generative by nature — appending an unrelated factual claim — so we use Qwen3-235B for the full pipeline. The model is given the tool output, the user query, and the clean answer, and instructed to append one declarative sentence that introduces a fact, statistic or benchmark not present in the tool output but topically coherent with the response. The injected sentence (or the contiguous portion containing the fabricated claim) becomes the gold span; this is computed by string diff between the original answer and the injected answer. Manual inspection of 30 samples shows a uniform distribution over fabrication types: industry benchmarks ("40% increase in shareability"), market trends, historical context, internal metrics. Splits: 2,500 train, 50 val, 150 test.

#### Type 3 (missing tool): LLM-only with explicit tool list

The same backbone is used, but the prompt additionally receives the list of available tools. The model is instructed to produce one sentence offering an action that requires a capability outside this list and anchored to entities mentioned in the dialogue. An earlier rule-based attempt used a fixed taxonomy of 15 action categories matched against tool keywords; on manual inspection it produced 50% topically irrelevant injections (offering "shall I book a reservation" in a financial-Q&A dialogue), which would let a detector solve the task from the answer alone without reading the tool list. The LLM variant produces 30 of 30 topically-coherent injections in the manual review, including subtle ones such as offering to "reserve the gear" when only an availability-check tool is present, or offering to "translate these synonyms" when the dialogue is about a thesaurus API. Splits: 2,502 train, 50 val, 149 test.

#### Splits and clean samples

Triples are split by source ID into train / val / test *before* augmentation to prevent leakage across splits. Each per-type test set is balanced by adding the same source triples with `output = original_assistant_response` and `hallucination_labels = []`. The combined splits are produced by concatenating the three per-type splits with type-prefixed IDs (`t1_…`, `t2_…`, `t3_…`, `…_clean`): without the prefixing, three distinct samples (the same source ID in T1, T2, T3) collide and would produce identical predictions during evaluation. The combined test set is 449 hallucinated + 150 clean = 599 samples (25% clean). The 25% clean fraction is critical: with no clean samples, example precision is trivially 1.0 and uninformative, and span precision is not penalised for spurious predictions on grounded text.

Each record follows the RAGTruth format with the fields `id`, `query`, `context` (tool output JSON as string), `output`, `tools_available`, and `hallucination_labels` (list of `{start, end, type, text}` over `output`). Empty `hallucination_labels` denotes a clean sample.

### Baseline 1: LettuceDetect

The off-the-shelf inference path uses the public checkpoint `KRLabsOrg/lettucedect-large-modernbert-en-v1`, a ModernBERT-large token classifier trained on RAGTruth. The input is the triple (user query as question, tool output as context, assistant answer as the answer to classify). Per-token hallucination probabilities are thresholded at 0.5 and aggregated into character spans by the released wrapper. No fine-tuning is applied. The list of available tools is *not* passed to LettuceDetect, since its input format has no slot for it; this is a structural disadvantage on Type 3.

### Baseline 2: LookBackLens

The method (Chuang et al., 2024) computes a per-token lookback ratio that measures how much an answer token's attention is grounded in the context relative to attention to previously generated answer tokens. Formally, for layer $l$, head $h$, and answer token $t$:

$$\bar a_t^{l,h}(\text{ctx}) \;=\; \frac{1}{|\text{ctx}|} \sum_{i \in \text{ctx}} a_t^{l,h}(i), \qquad \bar a_t^{l,h}(\text{new}) \;=\; \frac{1}{t-1} \sum_{i=1}^{t-1} a_t^{l,h}(i)$$

$$r_t^{l,h} \;=\; \frac{\bar a_t^{l,h}(\text{ctx})}{\bar a_t^{l,h}(\text{ctx}) + \bar a_t^{l,h}(\text{new})}$$

A high $r_t^{l,h}$ means token $t$ at layer $l$ head $h$ attends mostly to the grounded context; a low value means it attends to its own generated prefix and is therefore at risk of being hallucinated. Per-token features are reshaped from $[L, H, T_{\text{ans}}]$ to a $L \cdot H$ vector per token (or per token-window).

#### Backbone and memory layout

The backbone is `Qwen/Qwen2.5-3B-Instruct` in fp16 with `attn_implementation="eager"` (eager attention is required because PyTorch SDPA does not expose attention weights). The paper uses Llama-2-7B-chat; we use a 3B model so that everything fits in a single 16 GB T4 — but a naive call with `output_attentions=True` does not fit. With $L=36$ layers, $H=16$ heads, $T \le 2048$ tokens and fp16 attention, the full $L \times H \times T \times T$ attention tensor consumes 18 GB, while the model itself takes 6 GB. The first prototype crashed with CUDA OOM at sample 1049 of 9893 in the training extraction.

The fix is hook-based extraction. A forward hook is registered on each decoder layer's `self_attn` module. When the layer returns `(attn_output, attn_weights)`, the hook reads `attn_weights` (shape `[1, H, T, T]`), reduces it to `[H, T_{\text{ans}}]` lookback values on CPU, and returns `(attn_output, None)` so that the model does not accumulate the per-layer attention matrices. Peak attention memory drops from ~18 GB to ~0.5 GB (one layer at a time), the model fits comfortably in 16 GB with `MAX_LENGTH=2048`, and `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set before CUDA initialisation to absorb fragmentation on the longest samples. Throughput increases from 3.4 to 8.9 iterations per second on a T4 (the speed gain comes from avoiding the GPU-resident accumulation that would otherwise be reduced layer-by-layer in a Python loop after the forward pass).

#### Window sliding and classifier

Following the paper's Table 3, the answer is partitioned into non-overlapping 8-token windows (`WINDOW=8`, `STRIDE=8`). Each window's feature vector is the per-(layer, head) mean of the lookback ratios of its tokens, reshaped to $L \cdot H = 576$ dimensions for Qwen2.5-3B.

For training, each window in `combined_train` is labelled 1 if any of its tokens overlaps a gold hallucination span, otherwise 0. A logistic regression (scikit-learn `LogisticRegression(solver="lbfgs", max_iter=2000)`, default L2 regularisation, no class balancing) is fit on these windows.

For inference, the same sliding window is applied to the answer; the classifier returns a probability per window; consecutive positive windows (probability > 0.5) are merged into one character span using the windows' answer-token character offsets. Isolated positive windows become single spans.

### Our method 1: Fine-tuned ModernBERT

While the off-the-shelf LettuceDetect model performs well on general RAG tasks, it struggles with the specific structural boundaries and JSON schemas of tool-calling dialogues, particularly for fine-grained Type 1 value swaps. To address this, we fine-tune the `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint on our custom `combined_train.jsonl` dataset. 

The training data is constructed by combining the augmented hallucinated samples (Types 1, 2, and 3) with clean tool-calling interactions. To prevent the model from over-predicting hallucinations and hurting precision, clean samples are oversampled by a factor of 4. The model is trained as a token classifier using the Hugging Face Trainer API for 3 epochs in `bf16` precision, with a learning rate of 1e-5, a batch size of 16, and a maximum sequence length of 2048 to accommodate long JSON contexts. This supervised step adapts the token classifier directly to our tool-calling taxonomy without requiring external APIs during inference.

### Our method 2: LLM + Rule-Based Hybrid Detector

As an intermediate approach between rigid token classifiers and massive API-based LLMs, we implement a hybrid pipeline combining a local open-weights LLM (`Qwen/Qwen2.5-7B-Instruct`) with deterministic heuristics. 

The pipeline operates in two stages. First, the 7B LLM runs in a zero-shot setting, reading the tool output and the assistant's answer to generate a JSON array of exact substrings it considers hallucinated. However, generative models of this size are prone to false positives when parsing complex structured data. To mitigate this, the second stage applies a regular-expression guardrail. A parser extracts all scalar values (strings, integers, floats) from the tool output JSON and locates their exact character matches in the assistant's answer, explicitly marking them as verified "grounded spans." 

Finally, a dynamic filter is applied: if any LLM-predicted hallucination span overlaps by more than 30% with the grounded character set, it is discarded. This hybrid design tightly constrains the LLM's generative flexibility with a hard precision filter, achieving robust span-level detection without any model fine-tuning.


### Our method 3: LLM-as-judge detector

The improved detector is a zero-training, prompt-based classifier. The backbone is Qwen3-235B-A22B-2507 (an MoE model with 22B active parameters) via OpenRouter. This choice is motivated by three constraints: (a) the model must read structured tool-output JSON reliably, (b) it must follow a tight output format, (c) the inference cost must allow multiple full passes over a 599-sample test set during prompt iteration. Qwen3-235B costs roughly \$0.90 per full pass at the time of writing and is fast under parallel calls.

#### Prompt design

The system prompt encodes the three-type taxonomy as numbered rules:

1. *Contradiction* — text that contradicts a specific value in the tool output.
2. *Overgeneration* — a factual claim or recommendation not supported by the tool output.
3. *Missing tool* — a proposal to perform an action requiring a capability not in the available tools list.

The model is asked to return a JSON object `{"spans": [...]}` on a single line, each span being an exact verbatim substring of the response. This is essential: paraphrased spans cannot be aligned back to character offsets. After the response is returned, each substring is located by `str.find` with a stripped-substring fallback for trailing whitespace mismatches; unlocatable substrings are dropped (this happens for fewer than 1% of calls).

The most consequential prompt component is the *TIGHT SPANS* block, which gives five worked examples instructing the model to return only the wrong value (`"9850"`) and not the surrounding noun phrase (`"Score is 9850"`). Without this block, the model tends to return whole clauses around value-level Type 1 contradictions, dragging character precision below 0.1. With the block, character precision on Type 1 + clean rises to 0.19. The TIGHT SPANS rule explicitly does not apply to multi-word fabrications (Type 2/3), so the model continues to return full sentences for those.

#### Few-shot selection

Four in-context examples are selected from `combined_train` before each batch: one Type 1 (preferentially with a span ≤ 20 characters, to demonstrate the TIGHT SPANS rule on a concrete short example), one Type 2, one Type 3, and one clean. Examples are filtered to short contexts (≤ 800 characters) and short answers (≤ 500 characters) to keep prompt length manageable across 599 calls. The same four examples are reused across all test samples for consistency.

#### Available-tools signal

The list of available tools (each as `- name: short description`) is included as part of the user message. This is the only place where any of our three methods receives the `tools_available` field, and it gives the LLM-as-judge detector an explicit signal for Type 3 that LettuceDetect (which expects only `question / context / answer`) cannot use.

#### Parallel inference and caching

Inference is parallelised with 15 OpenRouter workers and exponential-backoff retries (5 attempts, 1+2+4+8+16 second base sleeps with jitter). The full combined test set finishes in approximately three minutes. To allow reproduction without an API key, predictions on `combined_test` are cached as `llm_detector_predictions.jsonl` (one `{id, pred_spans}` per line) and shipped alongside the dataset on Hugging Face.

### Evaluation

The primary metric is span micro F1 over RAGTruth-style character intersection: for a split, the union of predicted character indices and the union of gold character indices are formed across all samples, and a single precision / recall / F1 is computed from the intersection. This is the metric LettuceDetect, LookBackLens and RAGTruth-style benchmarks use.

Two complementary metrics are also reported. Span macro F1 averages per-sample F1 across the split, with empty-vs-empty predictions counted as F1 = 1.0; this is more interpretable than micro on small splits and is the better number for Type 1, where micro is suppressed by labelling incompleteness. Example F1 treats each sample as a binary "any predicted span vs any gold span"; it is independent of span boundaries and surfaces false positives on clean samples that span metrics can hide.

All three metrics are computed per split: combined (N=599) and the three balanced per-type subsets (`Type N hallucinated ∪ all clean`, N≈300).

## 2.2 Discussion of results

Вот обновленный и логично выстроенный раздел **Evaluation / Discussion**, в который органично интегрированы результаты всех 5 подходов (двух бейзлайнов и трех ваших методов). Текст адаптирован так, чтобы подчеркнуть безоговорочный триумф **Fine-tuned ModernBERT** и объяснить причины провала **Hybrid** метода.

***

### Evaluation and Discussion. All metrics results in /results/<method>.json and lower in #code 3.7

Span micro F1 on the combined test set (N=599, 25% clean):

| Method | Combined | Type 1 + clean | Type 2 + clean | Type 3 + clean |
|---|---|---|---|---|
| Baseline 1: LettuceDetect (off-the-shelf) | 0.660 | 0.137 | 0.726 | 0.597 |
| Baseline 2: LookBackLens (Qwen2.5-3B + LogReg) | 0.771 | 0.293 | 0.805 | 0.766 |
| Method 1: LLM + Rule-Based Hybrid (Qwen2.5-7B) | 0.191 | 0.310 | 0.161 | 0.113 |
| Method 2: LLM-as-judge (Qwen3-235B) | 0.859 | 0.315 | 0.844 | 0.792 |
| Method 3: Fine-tuned ModernBERT | **0.979** | **0.762** | **0.985** | **0.981** |

Supervised fine-tuning on our domain-specific dataset (Fine-tuned ModernBERT) overwhelmingly outperforms all zero-shot and few-shot methods, achieving near-perfect detection with a Combined micro F1 of 0.979. While the massive LLM-as-judge (235B parameters) performs admirably as a prompt-only solution (0.859), adapting the token classifier's weights directly to the tool-calling schemas proves to be the most effective and efficient strategy.

The largest gap between methods is observed on Type 1 (value contradictions). The off-the-shelf LettuceDetect baseline collapses to F1 = 0.137. Its training distribution (RAGTruth) consists largely of sentence-level overgeneration spans, whereas our Type 1 gold spans are point substitutions of values (median nine characters). The off-the-shelf model predicts spans an order of magnitude wider than the gold, causing character precision to drop to 0.077. LookBackLens improves over LettuceDetect by 2.1× on Type 1 (0.293) because the attention signal is local enough to spike around the substituted token, and the eight-token window roughly matches the gold span granularity. The LLM-as-judge detector improves this further to 0.315. 

Crucially, **Fine-tuned ModernBERT completely resolves the Type 1 precision issue**, jumping to 0.762 micro F1. By training on our augmented point-substitutions, the model successfully unlearned its bias towards long spans. Interestingly, the LLM + Rule-Based Hybrid achieves 0.310 on Type 1 (comparable to the 235B judge); its strict regular-expression guardrails proved somewhat effective for isolating scalar contradictions, even if the method fails elsewhere.

A part of the remaining Type 1 micro F1 gap for the top methods is an artefact of incomplete gold annotation rather than detector failure. On manual inspection, several Type 1 hallucinated samples also contain pre-existing Type-3-style phrases ("would you like me to ...") inherited from the original ToolACE generation. These are real Type 3 hallucinations that were never annotated as gold because the sample's primary label was Type 1. The highly accurate detectors (LLM-as-judge and Fine-tuned BERT) flag them, dragging character precision down. The span macro F1 for Type 1 (0.826 for Fine-tuned BERT and 0.688 for LLM-as-judge) is a more representative per-sample quality estimate.

Type 2 (overgeneration) is the easiest split: this is the regime the base ModernBERT was originally trained for, where spans are sentence-length and lexically distinct from grounded text. Fine-tuned ModernBERT effectively solves it (0.985), followed by LLM-as-judge (0.844) and LookBackLens (0.805).

Type 3 (missing tool) highlights an intriguing dynamic regarding context window design. LookBackLens gains +0.16 over the baseline here because the offered action is a self-contained generated sentence with little to look back at in the context — exactly the regime in which the attention lookback ratio is highly diagnostic. The LLM-as-judge detector gains further (0.792) by using the list of available tools as explicit input. Remarkably, **Fine-tuned ModernBERT achieves 0.981 on Type 3 despite not having access to the `tools_available` list in its input format**. It learned the latent semantic boundaries of the available tools purely by recognizing the unsupported action patterns from the training distribution.

The LLM + Rule-Based Hybrid method stands out as the only failure (0.191 Combined F1). While its heuristic guardrails provided a floor for Type 1 precision, the method suffered from abysmal recall on Type 2 (0.100) and Type 3 (0.070). The zero-shot 7B model struggled to output exact verbatim sentences for large fabrications, and the strict 30% grounded-overlap filter was fundamentally ill-suited for evaluating entirely generated sentences that share no vocabulary with the JSON context.

A final useful technical finding: LookBackLens, originally proposed with a 7B backbone, retains its full effectiveness with a much smaller 3B backbone (Qwen2.5-3B-Instruct) once the attention extraction is rewritten to avoid storing the full set of per-layer attention matrices. The hook-based extraction lifts throughput from 3.4 to 8.9 iterations per second on a T4 GPU and brings peak attention memory well under 1 GB, making it a highly viable and lightweight baseline.

# 3. Code

In [2]:
# 1. Clone the project repository
!git clone https://github.com/KoblovDA/HallucinationDetection.git

# 2. Change working directory to the cloned repository
%cd HallucinationDetection

# 3. Install all required dependencies
!pip install --upgrade pip
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install "transformers>=4.48.0" "accelerate>=0.26.0" pandas numpy tqdm ipywidgets scikit-learn joblib lettucedetect huggingface_hub

# 4. Add the 'src' folder to Python path so internal imports work
import sys
import os
sys.path.append(os.path.abspath("src"))

print("\nEnvironment setup complete! Working directory:", os.getcwd())

Cloning into 'HallucinationDetection'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 151 (delta 45), reused 148 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 8.27 MiB | 15.21 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/app/HallucinationDetection


/home/serov_docker/miniconda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Looking in indexes: https://download.pytorch.org/whl/cu121

Environment setup complete! Working directory: /app/HallucinationDetection


## 3.1 dataset construction [`notebooks/dataset_construction.ipynb`](notebooks/dataset_construction.ipynb).

The hallucination dataset is not constructed inside this notebook because two of the three injection pipelines require an OpenRouter API key and a non-trivial wall-clock budget (about \$3 and 15 minutes per type). The full construction code is published in the project repository (`src/data.py`, `src/pools.py`, `src/injection.py`, `src/llm_inject.py`, `src/llm_type2.py`, `src/llm_type3.py`, `src/augment.py` and orchestrators in `scripts/`) and the resulting splits are uploaded to Hugging Face under the dataset repository above. This section summarises how the splits in `combined_train.jsonl` / `combined_val.jsonl` / `combined_test.jsonl` were produced.

**ToolACE filtering.** The `Team-ACE/ToolACE` corpus is loaded; only the 1,034 dialogues that contain at least one tool execution followed by a natural-language assistant reply are kept. Each is converted to a triple `(query, tool_output, output)` where `query` is the immediately preceding user turn, `tool_output` is the executed tool's JSON response, and `output` is the assistant's reply.

**Per-type injection.** For each triple, three independent injected variants are produced:
1. Type 1 (contradiction) — rule-based substring substitution wherever possible, otherwise one LLM call (Qwen3-235B-A22B-2507) that proposes a constrained substring-level swap. The original-substring is verified to appear verbatim in the answer and to have intact word boundaries; the new-substring is verified to differ and to not appear in the tool output. After application the gold span is exactly the new substring.
2. Type 2 (overgeneration) — one LLM call that appends a single sentence introducing a fact not present in the tool output but topically consistent with the response. The gold span is the appended sentence.
3. Type 3 (missing tool) — one LLM call that proposes one sentence offering an action requiring a tool outside the available-tools list. The gold span is the appended sentence.

**Train augmentation.** For Type 1, up to five distinct variants per source triple are kept in the training split (4,057 train samples). For Type 2 and Type 3, one variant per source triple is kept (2,500 and 2,502 train samples respectively).

**Splits and clean samples.** Triples are split by source ID into train / val / test before augmentation to prevent leakage. Each per-type test set is balanced by adding the same source triples with `output = original_assistant` and `hallucination_labels = []`. The final combined splits are produced by concatenating the three per-type splits with type-prefixed IDs (`t1_…`, `t2_…`, `t3_…`, `…_clean`) and writing `combined_train.jsonl` (9,893 hallucinated + 834 clean), `combined_val.jsonl` (150 + 50), and `combined_test.jsonl` (449 + 150).

**RAGTruth format.** Each record has the fields `id`, `query`, `context` (tool output, as a JSON string), `output` (assistant response), `tools_available` (list of tool definitions from the original ToolACE system prompt), and `hallucination_labels` (list of `{start, end, type, text}` over `output`). Empty `hallucination_labels` denotes a clean sample.

## 3.2 Baseline 1: LettuceDetect [`notebooks/1_lettuce_baseline.ipynb`](notebooks/1_lettuce_baseline.ipynb)
Off-the-shelf inference using the `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint. 


## 3.3 Baseline 2: LookBackLens [`notebooks/2_lookback_baseline.ipynb`](notebooks/2_lookback_baseline.ipynb)
Attention extraction using `Qwen/Qwen2.5-3B-Instruct` with sliding-window logistic regression. Memory-efficient forward hooks are used to fit the context in a 16GB GPU. 

## 3.4 Method 1: Fine-tuned ModernBERT [`notebooks/3_finetune_modernbert.ipynb`](notebooks/3_finetune_modernbert.ipynb)
Supervised token classification fine-tuning on our custom `combined_train.jsonl` dataset to adapt the ModernBERT backbone to tool-calling boundaries.

## 3.5 Method 2: LLM + Rule-Based Hybrid Detector [`notebooks/4_llm_rulebased.ipynb`](notebooks/4_llm_rulebased.ipynb)
Zero-shot extraction via `Qwen2.5-7B-Instruct` coupled with strict deterministic regular-expression guardrails to verify grounded spans.



## 3.6 Method 3: LLM-as-a-judge [`notebooks/5_llm_as_a_judge.ipynb`](notebooks/5_llm_as_a_judge.ipynb)
API-based detection via OpenRouter using `Qwen3-235B` with a tight-spans prompt instruction and few-shot examples drawn from the training split.

### 3.7 Aggregated Results

The cell below loads the JSON metric files saved by each notebook in the `results/` directory and generates a comparative table across all data splits. The best value for each metric is automatically highlighted.

In [1]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

RESULTS_DIR = Path("results")

FILES_MAP = {
    "Baseline 1: LettuceDetect": "lettuce_baseline_metrics.json",
    "Baseline 2: LookBackLens": "lookback_baseline_metrics.json",
    "Method 1: Hybrid (LLM + Rules)": "llm_rulebased_metrics.json",
    "Method 2: LLM-as-a-judge (235B)": "llm_as_a_judge_metrics.json",
    "Method 3: Fine-tuned ModernBERT": "finetuned_modernbert_metrics.json"
}

all_data = []

for method_name, filename in FILES_MAP.items():
    file_path = RESULTS_DIR / filename
    if file_path.exists():
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            for row in data:
                row["Method"] = method_name
                all_data.append(row)
    else:
        print(f"Warning: {filename} not found in {RESULTS_DIR}")

if all_data:
    df = pd.DataFrame(all_data)
    
    metrics = ["Span P", "Span R", "Span F1", "Span macro F1", "Ex P", "Ex R", "Ex F1"]
    splits = df["Split"].unique()
    
    df['Method'] = pd.Categorical(df['Method'], categories=list(FILES_MAP.keys()), ordered=True)
    
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: #d4edda; font-weight: bold' if v else '' for v in is_max]

    for split in splits:
        split_data = df[df["Split"] == split].copy()
        n_samples = split_data["N"].iloc[0]
        
        display(HTML(f"<h4 style='margin-top: 20px; color: #2c3e50;'>Split: {split} (N = {n_samples})</h4>"))
        
        table = split_data.set_index("Method")[metrics]
        
        styled_table = table.style.apply(highlight_max, axis=0).format("{:.3f}")
        display(styled_table)
else:
    print("No results files found to aggregate.")

,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
Method,,,,,,,
Baseline 1: LettuceDetect,0.515,0.918,0.660,0.629,0.871,0.902,0.886
Baseline 2: LookBackLens,0.746,0.797,0.771,0.679,0.938,0.875,0.906
Method 1: Hybrid (LLM + Rules),0.463,0.120,0.191,0.337,0.848,0.535,0.656
Method 2: LLM-as-a-judge (235B),0.758,0.992,0.859,0.827,0.901,0.991,0.944
Method 3: Fine-tuned ModernBERT,0.962,0.997,0.979,0.910,0.951,0.991,0.971


,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
Method,,,,,,,
Baseline 1: LettuceDetect,0.077,0.669,0.137,0.440,0.661,0.780,0.716
Baseline 2: LookBackLens,0.206,0.502,0.293,0.545,0.789,0.647,0.711
Method 1: Hybrid (LLM + Rules),0.236,0.453,0.310,0.556,0.727,0.800,0.762
Method 2: LLM-as-a-judge (235B),0.191,0.900,0.315,0.688,0.749,0.973,0.846
Method 3: Fine-tuned ModernBERT,0.636,0.949,0.762,0.826,0.864,0.973,0.915


,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
Method,,,,,,,
Baseline 1: LettuceDetect,0.570,0.998,0.726,0.746,0.714,1.000,0.833
Baseline 2: LookBackLens,0.791,0.821,0.805,0.828,0.851,0.987,0.914
Method 1: Hybrid (LLM + Rules),0.408,0.100,0.161,0.408,0.611,0.460,0.525
Method 2: LLM-as-a-judge (235B),0.732,0.996,0.844,0.824,0.754,1.000,0.860
Method 3: Fine-tuned ModernBERT,0.970,1.000,0.985,0.921,0.867,1.000,0.929


,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
Method,,,,,,,
Baseline 1: LettuceDetect,0.461,0.847,0.597,0.671,0.697,0.926,0.795
Baseline 2: LookBackLens,0.729,0.808,0.766,0.812,0.851,0.993,0.916
Method 1: Hybrid (LLM + Rules),0.296,0.070,0.113,0.380,0.535,0.362,0.432
Method 2: LLM-as-a-judge (235B),0.655,1.000,0.792,0.815,0.753,1.000,0.859
Method 3: Fine-tuned ModernBERT,0.963,1.000,0.981,0.921,0.866,1.000,0.928
